# Phase 5 & 6 — Gold Layer (Star Schema) + Business KPI Reporting
**Apex Retail Intelligence | Celebal Technologies CEI'26 Major Project**

Builds the Gold-layer Star Schema (`dim_customer`, `dim_product`, `dim_promotion`, `dim_date`,
`fact_sales`), registers every table in Unity Catalog under `GOLD_tables`, and computes all
five required KPIs entirely with PySpark / Spark SQL — no external BI tools.

In [0]:
dbutils.widgets.text("catalog", "apex_retail1", "Catalog")
dbutils.widgets.text("silver_schema", "silver_tables", "Silver Schema")
dbutils.widgets.text("gold_schema", "GOLD_tables", "Gold Schema (Unity Catalog)")

CATALOG = dbutils.widgets.get("catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

from pyspark.sql import functions as F, Window

silver_customer = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_customer")
silver_product = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_product")
silver_sales = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.fact_sales_ledger")

## 5.1 Star Schema Construction

In [0]:
# ============================================================
# GOLD LAYER - STAR SCHEMA SETUP
# Serverless-compatible
# ============================================================

# ------------------------------------------------------------
# 1. WIDGETS
# ------------------------------------------------------------

dbutils.widgets.text(
    "catalog",
    "apex_retail1",
    "Catalog"
)

dbutils.widgets.text(
    "silver_schema",
    "silver_tables",
    "Silver Schema"
)

dbutils.widgets.text(
    "gold_schema",
    "GOLD_tables",
    "Gold Schema (Unity Catalog)"
)


# ------------------------------------------------------------
# 2. READ CONFIGURATION
# ------------------------------------------------------------

CATALOG = dbutils.widgets.get("catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")


# ------------------------------------------------------------
# 3. CREATE GOLD SCHEMA
# ------------------------------------------------------------

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{GOLD_SCHEMA}
    """
)

print("=" * 70)
print("GOLD LAYER INITIALIZATION")
print("=" * 70)

print(f"Catalog       : {CATALOG}")
print(f"Silver Schema : {SILVER_SCHEMA}")
print(f"Gold Schema   : {GOLD_SCHEMA}")


# ------------------------------------------------------------
# 4. LOAD SILVER TABLES
# ------------------------------------------------------------

silver_customer = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer"
)

silver_product = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dim_product"
)

silver_sales = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.fact_sales_ledger"
)


# ------------------------------------------------------------
# 5. BASIC VALIDATION
# ------------------------------------------------------------

customer_count = silver_customer.count()
product_count = silver_product.count()
sales_count = silver_sales.count()


print("\nSilver layer row counts:")
print(f"  dim_customer      : {customer_count}")
print(f"  dim_product       : {product_count}")
print(f"  fact_sales_ledger : {sales_count}")


# ------------------------------------------------------------
# 6. SHOW SILVER SCHEMAS
# ------------------------------------------------------------

print("\nDIM_CUSTOMER schema:")
silver_customer.printSchema()

print("\nDIM_PRODUCT schema:")
silver_product.printSchema()

print("\nFACT_SALES_LEDGER schema:")
silver_sales.printSchema()


# ------------------------------------------------------------
# 7. CREATE CLEAN PRODUCT DATAFRAME
#
# IMPORTANT:
# "Unknown" date values are converted to NULL instead
# of causing CAST_INVALID_INPUT.
# ------------------------------------------------------------

silver_product_clean = (
    silver_product
    .withColumn(
        "product_manufacture_date_clean",
        F.expr(
            "try_cast(product_manufacture_date AS DATE)"
        )
    )
    .withColumn(
        "product_expiry_date_clean",
        F.expr(
            "try_cast(product_expiry_date AS DATE)"
        )
    )
)


# ------------------------------------------------------------
# 8. REPORT INVALID DATE VALUES
# ------------------------------------------------------------

invalid_manufacture_dates = (
    silver_product
    .filter(
        F.col("product_manufacture_date").isNotNull()
        &
        (
            F.expr(
                "try_cast(product_manufacture_date AS DATE)"
            ).isNull()
        )
    )
    .count()
)

invalid_expiry_dates = (
    silver_product
    .filter(
        F.col("product_expiry_date").isNotNull()
        &
        (
            F.expr(
                "try_cast(product_expiry_date AS DATE)"
            ).isNull()
        )
    )
    .count()
)


print("\nDate quality check:")
print(
    f"  Invalid manufacture dates : "
    f"{invalid_manufacture_dates}"
)

print(
    f"  Invalid expiry dates      : "
    f"{invalid_expiry_dates}"
)


# ------------------------------------------------------------
# 9. DISPLAY STAR SCHEMA COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STAR SCHEMA SOURCE COUNTS")
print("=" * 70)

print(
    f"dim_customer     : {customer_count}"
)

print(
    f"dim_product      : {product_count}"
)

# Promotion will be created in the next step
print(
    "dim_promotion    : To be created"
)

print(
    f"fact_sales_ledger: {sales_count}"
)

print("=" * 70)

print(
    "\nGold layer source tables loaded successfully."
)

GOLD LAYER INITIALIZATION
Catalog       : apex_retail1
Silver Schema : silver_tables
Gold Schema   : GOLD_tables

Silver layer row counts:
  dim_customer      : 3151
  dim_product       : 1041
  fact_sales_ledger : 2000

DIM_CUSTOMER schema:
root
 |-- customer_id: string (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: double (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: double (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- surrogate_key: string (nullable = true)
 |-- version: string (nullable = true)
 |-- effective_start_date: string (nullable = tr

## 5.2 Unity Catalog Registration

In [0]:
# ============================================================
# APEX RETAIL - GOLD STAR SCHEMA
# COMPLETE SERVERLESS-COMPATIBLE VERSION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 70)
print("GOLD LAYER - STAR SCHEMA BUILD")
print("=" * 70)

# ============================================================
# 1. PARAMETERS
# ============================================================

CATALOG = "apex_retail1"
SILVER_SCHEMA = "silver_tables"
GOLD_SCHEMA = "GOLD_tables"

GOLD_BASE = f"{CATALOG}.{GOLD_SCHEMA}"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {GOLD_BASE}"
)

print(f"Catalog       : {CATALOG}")
print(f"Silver Schema : {SILVER_SCHEMA}")
print(f"Gold Schema   : {GOLD_SCHEMA}")


# ============================================================
# 2. LOAD SILVER TABLES
# ============================================================

silver_customer = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer"
)

silver_product = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dim_product"
)

silver_sales = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.fact_sales_ledger"
)

print("\nSilver source counts:")
print(
    f"dim_customer      : {silver_customer.count()}"
)
print(
    f"dim_product       : {silver_product.count()}"
)
print(
    f"fact_sales_ledger : {silver_sales.count()}"
)


# ============================================================
# 3. GOLD DIM_CUSTOMER
# SCD TYPE 2 -> CURRENT RECORD ONLY
# ============================================================

gold_customer = (
    silver_customer
    .filter(
        F.col("is_current") == True
    )
    .withColumn(
        "effective_start_date",
        F.expr(
            "try_cast(effective_start_date AS DATE)"
        )
    )
    .withColumn(
        "effective_end_date",
        F.expr(
            "try_cast(effective_end_date AS DATE)"
        )
    )
)

# Remove accidental duplicate active customers
gold_customer = (
    gold_customer
    .dropDuplicates(["customer_id"])
)

print(
    f"\nGold dim_customer rows: "
    f"{gold_customer.count()}"
)


# ============================================================
# 4. GOLD DIM_PRODUCT
# SCD TYPE 1
# ============================================================

gold_product = (
    silver_product
    .dropDuplicates(["product_id"])
    .withColumn(
        "product_manufacture_date",
        F.expr(
            "try_cast(product_manufacture_date AS DATE)"
        )
    )
    .withColumn(
        "product_expiry_date",
        F.expr(
            "try_cast(product_expiry_date AS DATE)"
        )
    )
)

print(
    f"Gold dim_product rows: "
    f"{gold_product.count()}"
)


# ============================================================
# 5. GOLD DIM_PROMOTION
# ============================================================

promotion_base = (
    silver_sales
    .select(
        "promotion_id",
        "promotion_type"
    )
    .filter(
        F.col("promotion_id").isNotNull()
    )
    .dropDuplicates(
        ["promotion_id"]
    )
)

# Generate promotion surrogate key
# No JVM access - works on Serverless
promotion_window = Window.orderBy(
    "promotion_id"
)

gold_promotion = (
    promotion_base
    .withColumn(
        "promotion_sk",
        F.row_number().over(
            promotion_window
        )
    )
)

print(
    f"Gold dim_promotion rows: "
    f"{gold_promotion.count()}"
)


# ============================================================
# 6. CLEAN TRANSACTION DATE
# IMPORTANT:
# "Unknown" becomes NULL instead of causing CAST error
# ============================================================

sales_clean = (
    silver_sales
    .withColumn(
        "transaction_date_clean",
        F.expr(
            "try_cast(transaction_date AS DATE)"
        )
    )
)


# ============================================================
# 7. GOLD DIM_DATE
# ============================================================

date_values = (
    sales_clean
    .filter(
        F.col("transaction_date_clean").isNotNull()
    )
    .select(
        F.col(
            "transaction_date_clean"
        ).alias("full_date")
    )
    .dropDuplicates()
)


# Create date attributes
gold_date = (
    date_values
    .withColumn(
        "date_sk",
        F.date_format(
            "full_date",
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "year",
        F.year("full_date")
    )
    .withColumn(
        "quarter",
        F.quarter("full_date")
    )
    .withColumn(
        "month",
        F.month("full_date")
    )
    .withColumn(
        "month_name",
        F.date_format(
            "full_date",
            "MMMM"
        )
    )
    .withColumn(
        "week_of_year",
        F.weekofyear("full_date")
    )
    .withColumn(
        "day",
        F.dayofmonth("full_date")
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("full_date")
    )
    .withColumn(
        "day_name",
        F.date_format(
            "full_date",
            "EEEE"
        )
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("full_date").isin(
            [1, 7]
        )
    )
    .select(
        "date_sk",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week_of_year",
        "day",
        "day_of_week",
        "day_name",
        "is_weekend"
    )
)

print(
    f"Gold dim_date rows: "
    f"{gold_date.count()}"
)


# ============================================================
# 8. GOLD FACT SALES
# ============================================================

# ------------------------------------------------------------
# Customer lookup
# ------------------------------------------------------------

customer_lookup = (
    gold_customer
    .select(
        "customer_id",
        "customer_sk"
    )
    .dropDuplicates(
        ["customer_id"]
    )
)


# ------------------------------------------------------------
# Product lookup
# ------------------------------------------------------------

product_lookup = (
    gold_product
    .select(
        "product_id",
        "product_sk"
    )
    .dropDuplicates(
        ["product_id"]
    )
)


# ------------------------------------------------------------
# Promotion lookup
# ------------------------------------------------------------

promotion_lookup = (
    gold_promotion
    .select(
        "promotion_id",
        "promotion_sk"
    )
    .dropDuplicates(
        ["promotion_id"]
    )
)


# ------------------------------------------------------------
# Date lookup
# ------------------------------------------------------------

date_lookup = (
    gold_date
    .select(
        "full_date",
        "date_sk"
    )
)


# ============================================================
# JOIN SILVER FACT WITH GOLD DIMENSIONS
# ============================================================

gold_fact = (
    sales_clean.alias("s")

    # Customer
    .join(
        customer_lookup.alias("c"),
        F.col("s.customer_id")
        == F.col("c.customer_id"),
        "left"
    )

    # Product
    .join(
        product_lookup.alias("p"),
        F.col("s.product_id")
        == F.col("p.product_id"),
        "left"
    )

    # Promotion
    .join(
        promotion_lookup.alias("pr"),
        F.col("s.promotion_id")
        == F.col("pr.promotion_id"),
        "left"
    )

    # Date
    .join(
        date_lookup.alias("d"),
        F.col("s.transaction_date_clean")
        == F.col("d.full_date"),
        "left"
    )

    .select(
        # Fact keys
        F.col("s.sales_sk").alias(
            "sales_sk"
        ),

        F.col("s.transaction_id").alias(
            "transaction_id"
        ),

        # Dimension foreign keys
        F.col("c.customer_sk").alias(
            "customer_sk"
        ),

        F.col("p.product_sk").alias(
            "product_sk"
        ),

        F.col("pr.promotion_sk").alias(
            "promotion_sk"
        ),

        F.col("d.date_sk").alias(
            "date_sk"
        ),

        # Transaction attributes
        F.col(
            "s.transaction_date_clean"
        ).alias(
            "transaction_date"
        ),

        F.col("s.quantity").alias(
            "quantity"
        ),

        F.col("s.unit_price").alias(
            "unit_price"
        ),

        F.col("s.discount_applied").alias(
            "discount_applied"
        ),

        F.col("s.total_sales").alias(
            "total_sales"
        ),

        F.col("s.payment_method").alias(
            "payment_method"
        ),

        F.col("s.store_location").alias(
            "store_location"
        ),

        F.col("s.transaction_hour").alias(
            "transaction_hour"
        ),

        F.col("s.day_of_week").alias(
            "day_of_week"
        ),

        F.col("s.week_of_year").alias(
            "week_of_year"
        ),

        F.col("s.month_of_year").alias(
            "month_of_year"
        ),

        F.col("s.promotion_id").alias(
            "promotion_id"
        ),

        F.col("s.promotion_type").alias(
            "promotion_type"
        ),

        F.col("s.holiday_season").alias(
            "holiday_season"
        ),

        F.col("s.season").alias(
            "season"
        ),

        F.col("s.weekend").alias(
            "weekend"
        )
    )
)


# ============================================================
# 9. REMOVE DUPLICATE TRANSACTIONS
# ============================================================

gold_fact = (
    gold_fact
    .dropDuplicates(
        ["transaction_id"]
    )
)

print(
    f"Gold fact_sales rows: "
    f"{gold_fact.count()}"
)


# ============================================================
# 10. WRITE GOLD TABLES
# ============================================================

gold_tables = [
    ("dim_customer", gold_customer),
    ("dim_product", gold_product),
    ("dim_promotion", gold_promotion),
    ("dim_date", gold_date),
    ("fact_sales", gold_fact)
]


for name, df in gold_tables:

    full_name = (
        f"{GOLD_BASE}.{name}"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            full_name
        )
    )

    spark.sql(
        f"""
        COMMENT ON TABLE {full_name}
        IS 'Apex Retail Gold layer — Star Schema table, refreshed by 04_gold_layer_kpi.py'
        """
    )

    print(
        f"Registered {full_name}"
    )


# ============================================================
# 11. GOLD TABLE COUNTS
# ============================================================

print("\n" + "=" * 70)
print("GOLD STAR SCHEMA ROW COUNTS")
print("=" * 70)

for name, _ in gold_tables:

    full_name = (
        f"{GOLD_BASE}.{name}"
    )

    count = (
        spark.table(
            full_name
        ).count()
    )

    print(
        f"{name:<20} : {count}"
    )


# ============================================================
# 12. VALIDATION
# ============================================================

final_fact = spark.table(
    f"{GOLD_BASE}.fact_sales"
)


# Duplicate transactions
duplicate_transactions = (
    final_fact
    .groupBy("transaction_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


# NULL foreign keys
null_customer = (
    final_fact
    .filter(
        F.col("customer_sk").isNull()
    )
    .count()
)

null_product = (
    final_fact
    .filter(
        F.col("product_sk").isNull()
    )
    .count()
)

null_date = (
    final_fact
    .filter(
        F.col("date_sk").isNull()
    )
    .count()
)


# Duplicate dimension keys
customer_duplicates = (
    spark.table(
        f"{GOLD_BASE}.dim_customer"
    )
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

product_duplicates = (
    spark.table(
        f"{GOLD_BASE}.dim_product"
    )
    .groupBy("product_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


# ============================================================
# 13. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("GOLD STAR SCHEMA VALIDATION")
print("=" * 70)

print(
    f"Duplicate transaction IDs : "
    f"{duplicate_transactions}"
)

print(
    f"NULL customer_sk          : "
    f"{null_customer}"
)

print(
    f"NULL product_sk           : "
    f"{null_product}"
)

print(
    f"NULL date_sk              : "
    f"{null_date}"
)

print(
    f"Duplicate customer IDs    : "
    f"{customer_duplicates}"
)

print(
    f"Duplicate product IDs     : "
    f"{product_duplicates}"
)


# ============================================================
# 14. SHOW GOLD TABLES
# ============================================================

print("\n" + "=" * 70)
print("REGISTERED GOLD TABLES")
print("=" * 70)

spark.sql(
    f"""
    SHOW TABLES IN {GOLD_BASE}
    """
).display()


# ============================================================
# 15. FINAL STATUS
# ============================================================

if (
    duplicate_transactions == 0
    and customer_duplicates == 0
    and product_duplicates == 0
):

    print("\n" + "=" * 70)
    print("SUCCESS: GOLD STAR SCHEMA COMPLETE")
    print("=" * 70)

else:

    print("\nWARNING: GOLD VALIDATION REQUIRES ATTENTION")

print(
    f"\nGold location: {GOLD_BASE}"
)

GOLD LAYER - STAR SCHEMA BUILD
Catalog       : apex_retail1
Silver Schema : silver_tables
Gold Schema   : GOLD_tables

Silver source counts:
dim_customer      : 2101
dim_product       : 1041
fact_sales_ledger : 2000

Gold dim_customer rows: 1050
Gold dim_product rows: 1041


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold dim_promotion rows: 864
Gold dim_date rows: 996
Gold fact_sales rows: 2000
Registered apex_retail1.GOLD_tables.dim_customer
Registered apex_retail1.GOLD_tables.dim_product
Registered apex_retail1.GOLD_tables.dim_promotion
Registered apex_retail1.GOLD_tables.dim_date
Registered apex_retail1.GOLD_tables.fact_sales

GOLD STAR SCHEMA ROW COUNTS
dim_customer         : 1050
dim_product          : 1041
dim_promotion        : 864
dim_date             : 996
fact_sales           : 2000

GOLD STAR SCHEMA VALIDATION
Duplicate transaction IDs : 0
NULL customer_sk          : 0
NULL product_sk           : 1392
NULL date_sk              : 64
Duplicate customer IDs    : 0
Duplicate product IDs     : 0

REGISTERED GOLD TABLES


database,tableName,isTemporary
gold_tables,dim_customer,false
gold_tables,dim_date,false
gold_tables,dim_product,false
gold_tables,dim_promotion,false
gold_tables,fact_sales,false



SUCCESS: GOLD STAR SCHEMA COMPLETE

Gold location: apex_retail1.GOLD_tables


## Phase 6 — Business KPI Reporting
All five required KPIs, computed directly against the Gold Star Schema using PySpark
DataFrame operations and Spark SQL. Rendered inline in this notebook only — no external
dashboarding tools, per the assignment's explicit constraint.

In [0]:
fact = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_sales")
dcust = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_customer")
dprod = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_product")
dpromo = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_promotion")
ddate = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_date")

fact.createOrReplaceTempView("fact_sales")
dcust.createOrReplaceTempView("dim_customer")
dprod.createOrReplaceTempView("dim_product")
dpromo.createOrReplaceTempView("dim_promotion")
ddate.createOrReplaceTempView("dim_date")

### KPI 1 — Net Margin by Region
`store_location` is used as the regional grain. Net margin = gross revenue − discount value.

In [0]:
kpi1_net_margin_by_region = spark.sql("""
    SELECT
        store_location AS region,
        ROUND(SUM(total_sales), 2)                         AS gross_revenue,
        ROUND(SUM(total_sales * discount_applied), 2)       AS total_discount_value,
        ROUND(SUM(total_sales) - SUM(total_sales * discount_applied), 2) AS net_margin
    FROM fact_sales
    GROUP BY store_location
    ORDER BY net_margin DESC
""")
display(kpi1_net_margin_by_region)

region,gross_revenue,total_discount_value,net_margin
Location D,1284255.54,281025.84,1003229.7
Location B,1241078.01,278905.95,962172.06
Location C,1183512.33,262051.86,921460.47
Location A,1141196.75,278178.47,863018.28
Unknown,675559.03,136140.54,539418.49


### KPI 2 — Average Order Value (AOV) by Promotion Type

In [0]:
kpi2_aov_by_promotion = spark.sql("""
    SELECT
        p.promotion_type,
        ROUND(AVG(f.total_sales), 2) AS avg_order_value,
        COUNT(*)                     AS orders
    FROM fact_sales f
    LEFT JOIN dim_promotion p ON f.promotion_sk = p.promotion_sk
    GROUP BY p.promotion_type
    ORDER BY avg_order_value DESC
""")
display(kpi2_aov_by_promotion)

promotion_type,avg_order_value,orders
Flash Sale,3021.01,553
20% Off,2791.78,565
Buy One Get One Free,2718.49,592
Unknown,2304.42,290


### KPI 3 — Demographic Churn Heatmap (State × Loyalty Program)

In [0]:
kpi3_churn_heatmap = spark.sql("""
    SELECT
        customer_state,
        loyalty_program,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN churned = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
        ROUND(100.0 * SUM(CASE WHEN churned = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct
    FROM dim_customer
    WHERE is_current = true
    GROUP BY customer_state, loyalty_program
    ORDER BY churn_rate_pct DESC
""")
display(kpi3_churn_heatmap)

customer_state,loyalty_program,total_customers,churned_customers,churn_rate_pct
State X,Yes,161,89,55.28
State Z,Yes,161,84,52.17
State Y,Yes,180,92,51.11
State X,No,201,98,48.76
State Z,No,180,85,47.22
State Y,No,164,71,43.29
Old_State_2,No,1,0,0.00
State IL,No,1,0,0.00
Old_State_1,No,1,0,0.00


### KPI 4 — Product Quality Index (Return Rate by Category)

In [0]:
kpi4_product_quality_index = spark.sql("""
    SELECT
        product_category,
        ROUND(AVG(product_return_rate), 4) AS avg_return_rate,
        ROUND(AVG(product_rating), 2)       AS avg_rating,
        COUNT(*)                            AS num_products
    FROM dim_product
    GROUP BY product_category
    ORDER BY avg_return_rate DESC
""")
display(kpi4_product_quality_index)

product_category,avg_return_rate,avg_rating,num_products
Electronics,0.2674,3.01,181
Furniture,0.2669,3.06,212
Groceries,0.256,2.95,221
Clothing,0.2394,3.11,215
Toys,0.2329,2.9,212


### KPI 5 — Store Traffic by Hour (Busiest Hours & Days)

In [0]:
kpi5_store_traffic_by_hour = spark.sql("""
    SELECT
        transaction_hour,
        day_of_week,
        COUNT(*) AS transaction_count
    FROM fact_sales
    GROUP BY transaction_hour, day_of_week
    ORDER BY transaction_count DESC
""")
display(kpi5_store_traffic_by_hour)

transaction_hour,day_of_week,transaction_count
5,Wednesday,20
22,Thursday,19
5,Sunday,19
8,Thursday,18
7,Wednesday,18
2,Saturday,17
20,Saturday,17
22,Wednesday,17
1,Saturday,17
2,Sunday,17


## Persist KPI Outputs as Gold Tables (for downstream consumption / re-display)

In [0]:
kpi_tables = {
    "kpi_net_margin_by_region": kpi1_net_margin_by_region,
    "kpi_aov_by_promotion": kpi2_aov_by_promotion,
    "kpi_demographic_churn_heatmap": kpi3_churn_heatmap,
    "kpi_product_quality_index": kpi4_product_quality_index,
    "kpi_store_traffic_by_hour": kpi5_store_traffic_by_hour,
}
for name, df in kpi_tables.items():
    full_name = f"{CATALOG}.{GOLD_SCHEMA}.{name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    print(f"Saved KPI table {full_name}")

print("\n✅ Gold layer + all 5 KPIs complete. Pipeline finished end-to-end.")

Saved KPI table apex_retail1.GOLD_tables.kpi_net_margin_by_region
Saved KPI table apex_retail1.GOLD_tables.kpi_aov_by_promotion
Saved KPI table apex_retail1.GOLD_tables.kpi_demographic_churn_heatmap
Saved KPI table apex_retail1.GOLD_tables.kpi_product_quality_index
Saved KPI table apex_retail1.GOLD_tables.kpi_store_traffic_by_hour

✅ Gold layer + all 5 KPIs complete. Pipeline finished end-to-end.
